In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
data = pd.read_csv('titaniccsv/train.csv')
test = pd.read_csv('titaniccsv/test.csv')

In [6]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
data.drop(['PassengerId', 'Ticket', 'Cabin'], axis=1, inplace=True)

In [8]:
cols = ['SibSp', 'Parch', 'Age', 'Fare']
for col in cols:
    data[col].fillna(data[col].median(), inplace=True)
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_2368\2506033821.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_2368\2506033821.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [9]:
data.head(20)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S
5,0,3,"Moran, Mr. James",male,28.0,0,0,8.4583,Q
6,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,51.8625,S
7,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,21.0750,S
8,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,11.1333,S
9,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,30.0708,C


In [10]:
data["Sex"] = data["Sex"].replace({"male": 0, "female": 1})
data["Name"] = data["Name"].apply(lambda x: x.split(",")[1].split(".")[0].strip())

common_titles = ["Mr", "Mrs", "Miss", "Master"]
data["Name"] = data["Name"].apply(lambda t: t if t in common_titles else "Other")
data["Name"] = data["Name"].replace({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Other": 4})

C:\Users\Admin\AppData\Local\Temp\ipykernel_2368\4033265547.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["Sex"] = data["Sex"].replace({"male": 0, "female": 1})
C:\Users\Admin\AppData\Local\Temp\ipykernel_2368\4033265547.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["Name"] = data["Name"].replace({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Other": 4})


In [11]:
data.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,0,22.0,1,0,7.2500,S
1,1,1,1,1,38.0,1,0,71.2833,C
2,1,3,2,1,26.0,0,0,7.9250,S
3,1,1,1,1,35.0,1,0,53.1000,S
4,0,3,0,0,35.0,0,0,8.0500,S


In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['Embarked'] = le.fit_transform(data['Embarked'])

In [13]:
data["Family Size"] = data["SibSp"] + data["Parch"] + 1


In [14]:
data["IsAlone"] = data["Family Size"].apply(lambda x: 1 if x == 1 else 0)

In [15]:
# Calculate median age for each title
title_medians = data.groupby('Name')['Age'].median()

# Fill missing ages based on the person's title
data['Age'] = data.apply(
    lambda row: title_medians[row['Name']] if np.isnan(row['Age']) else row['Age'], 
    axis=1
)

In [16]:
y = data["Survived"]
x = data.drop("Survived", axis=1)

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 15],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters: {'bootstrap': True, 'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
Accuracy: 0.8268156424581006
Confusion Matrix:
[[92 13]
 [18 56]]
